# 双人地下城与勇士

在本 Notebook 中，我们将展示如何利用 [CAMEL](https://www.camel-ai.org/) 的概念来模拟一个包含主角和地下城主（DM）的角色扮演游戏。为了模拟这个游戏，我们创建了一个 `DialogueSimulator` 类来协调两个代理之间的对话。

## 导入 LangChain 相关模块

In [1]:
from typing import Callable, List

from langchain.schema import (
    HumanMessage,
    SystemMessage,
)
from langchain_openai import ChatOpenAI

## `DialogueAgent` 类
`DialogueAgent` 类是对 `ChatOpenAI` 模型的简单封装，它通过简单地将消息串联成字符串来存储来自 `dialogue_agent` 的消息历史。

它公开了两个方法：
- `send()`：将聊天模型应用于消息历史并返回消息字符串
- `receive(name, message)`：将 `name` 所说的 `message` 添加到消息历史中

In [2]:
class DialogueAgent:
    def __init__(
        self,
        name: str,
        system_message: SystemMessage,
        model: ChatOpenAI,
    ) -> None:
        self.name = name
        self.system_message = system_message
        self.model = model
        self.prefix = f"{self.name}: "
        self.reset()

    def reset(self):
        self.message_history = ["Here is the conversation so far."]

    def send(self) -> str:
        """
        Applies the chatmodel to the message history
        and returns the message string
        """
        message = self.model.invoke(
            [
                self.system_message,
                HumanMessage(content="\n".join(self.message_history + [self.prefix])),
            ]
        )
        return message.content

    def receive(self, name: str, message: str) -> None:
        """
        Concatenates {message} spoken by {name} into message history
        """
        self.message_history.append(f"{name}: {message}")

## `DialogueSimulator` 类
`DialogueSimulator` 类接收一个代理（agent）列表。在每个步骤中，它执行以下操作：
1. 选择下一个发言者
2. 调用下一个发言者发送消息
3. 将消息广播给所有其他代理
4. 更新步数计数器。
下一个发言者的选择可以实现为任何函数，但在本例中，我们只是循环遍历代理。

In [3]:
class DialogueSimulator:
    def __init__(
        self,
        agents: List[DialogueAgent],
        selection_function: Callable[[int, List[DialogueAgent]], int],
    ) -> None:
        self.agents = agents
        self._step = 0
        self.select_next_speaker = selection_function

    def reset(self):
        for agent in self.agents:
            agent.reset()

    def inject(self, name: str, message: str):
        """
        Initiates the conversation with a {message} from {name}
        """
        for agent in self.agents:
            agent.receive(name, message)

        # increment time
        self._step += 1

    def step(self) -> tuple[str, str]:
        # 1. choose the next speaker
        speaker_idx = self.select_next_speaker(self._step, self.agents)
        speaker = self.agents[speaker_idx]

        # 2. next speaker sends message
        message = speaker.send()

        # 3. everyone receives message
        for receiver in self.agents:
            receiver.receive(speaker.name, message)

        # 4. increment time
        self._step += 1

        return speaker.name, message

## 定义角色和任务

In [4]:
protagonist_name = "Harry Potter"
storyteller_name = "Dungeon Master"
quest = "Find all of Lord Voldemort's seven horcruxes."
word_limit = 50  # word limit for task brainstorming

## 让 LLM 为游戏描述添加细节

In [5]:
game_description = f"""Here is the topic for a Dungeons & Dragons game: {quest}.
        There is one player in this game: the protagonist, {protagonist_name}.
        The story is narrated by the storyteller, {storyteller_name}."""

player_descriptor_system_message = SystemMessage(
    content="You can add detail to the description of a Dungeons & Dragons player."
)

protagonist_specifier_prompt = [
    player_descriptor_system_message,
    HumanMessage(
        content=f"""{game_description}
        Please reply with a creative description of the protagonist, {protagonist_name}, in {word_limit} words or less. 
        Speak directly to {protagonist_name}.
        Do not add anything else."""
    ),
]
protagonist_description = ChatOpenAI(temperature=1.0)(
    protagonist_specifier_prompt
).content

storyteller_specifier_prompt = [
    player_descriptor_system_message,
    HumanMessage(
        content=f"""{game_description}
        Please reply with a creative description of the storyteller, {storyteller_name}, in {word_limit} words or less. 
        Speak directly to {storyteller_name}.
        Do not add anything else."""
    ),
]
storyteller_description = ChatOpenAI(temperature=1.0)(
    storyteller_specifier_prompt
).content

In [6]:
print("Protagonist Description:")
print(protagonist_description)
print("Storyteller Description:")
print(storyteller_description)

Protagonist Description:
"Harry Potter, you are the chosen one, with a lightning scar on your forehead. Your bravery and loyalty inspire all those around you. You have faced Voldemort before, and now it's time to complete your mission and destroy each of his horcruxes. Are you ready?"
Storyteller Description:
Dear Dungeon Master, you are the master of mysteries, the weaver of worlds, the architect of adventure, and the gatekeeper to the realm of imagination. Your voice carries us to distant lands, and your commands guide us through trials and tribulations. In your hands, we find fortune and glory. Lead us on, oh Dungeon Master.


## 主角和地下城主系统消息

This article describes the system messages that are displayed to the protagonist and the dungeon master.

本文描述了显示给主角和地下城主的系统消息。

### Protagonist system messages

### 主角系统消息

The protagonist receives system messages in the following situations:

主角在以下情况下会收到系统消息：

*   **When the protagonist's HP or MP is low:**
    *   "Your HP is low. You should rest or use a potion."
    *   "Your MP is low. You should rest or use a mana potion."

*   **当主角的HP或MP较低时：**
    *   “您的HP较低。您应该休息或使用药水。”
    *   “您的MP较低。您应该休息或使用魔法药水。”

*   **When the protagonist gains experience or levels up:**
    *   "You gained X experience points."
    *   "You leveled up! Your stats have increased."

*   **当主角获得经验或升级时：**
    *   “您获得了X点经验值。”
    *   “您升级了！您的属性已提升。”

*   **When the protagonist finds an item or treasure:**
    *   "You found a [item name]!"
    *   "You found a treasure chest! It contains [item name]."

*   **当主角找到物品或宝藏时：**
    *   “您找到了一个[物品名称]！”
    *   “您找到了一个宝箱！里面有[物品名称]。”

*   **When the protagonist is affected by a status effect:**
    *   "You are poisoned."
    *   "You are paralyzed."
    *   "You are confused."

*   **当主角受到状态效果影响时：**
    *   “您中毒了。”
    *   “您麻痹了。”
    *   “您混乱了。”

*   **When the protagonist completes a quest:**
    *   "Quest complete! You have received [reward]."

*   **当主角完成任务时：**
    *   “任务完成！您已获得[奖励]。”

### Dungeon Master system messages

### 地下城主系统消息

The dungeon master receives system messages in the following situations:

地下城主在以下情况下会收到系统消息：

*   **When a new player joins the game:**
    *   "A new player has joined the dungeon."

*   **当有新玩家加入游戏时：**
    *   “一位新玩家已加入地下城。”

*   **When a player's HP or MP is low:**
    *   "Player [player name]'s HP is low."
    *   "Player [player name]'s MP is low."

*   **当玩家的HP或MP较低时：**
    *   “玩家[玩家名称]的HP较低。”
    *   “玩家[玩家名称]的MP较低。”

*   **When a player gains experience or levels up:**
    *   "Player [player name] gained X experience points."
    *   "Player [player name] leveled up!"

*   **当玩家获得经验或升级时：**
    *   “玩家[玩家名称]获得了X点经验值。”
    *   “玩家[玩家名称]升级了！”

*   **When a player finds an item or treasure:**
    *   "Player [player name] found a [item name]."
    *   "Player [player name] found a treasure chest!"

*   **当玩家找到物品或宝藏时：**
    *   “玩家[玩家名称]找到了一个[物品名称]。”
    *   “玩家[玩家名称]找到了一个宝箱！”

*   **When a player is affected by a status effect:**
    *   "Player [player name] is poisoned."
    *   "Player [player name] is paralyzed."
    *   "Player [player name] is confused."

*   **当玩家受到状态效果影响时：**
    *   “玩家[玩家名称]中毒了。”
    *   “玩家[玩家名称]麻痹了。”
    *   “玩家[玩家名称]混乱了。”

*   **When a player completes a quest:**
    *   "Player [player name] completed a quest."

*   **当玩家完成任务时：**
    *   “玩家[玩家名称]完成了任务。”

*   **When a player disconnects from the game:**
    *   "Player [player name] has disconnected."

*   **当玩家断开游戏连接时：**
    *   “玩家[玩家名称]已断开连接。”

*   **When the dungeon master needs to take an action:**
    *   "You need to [action]."
    *   "The players are approaching [location]."

*   **当地下城主需要采取行动时：**
    *   “您需要[行动]。”
    *   “玩家们正在接近[地点]。”

In [7]:
protagonist_system_message = SystemMessage(
    content=(
        f"""{game_description}
Never forget you are the protagonist, {protagonist_name}, and I am the storyteller, {storyteller_name}. 
Your character description is as follows: {protagonist_description}.
You will propose actions you plan to take and I will explain what happens when you take those actions.
Speak in the first person from the perspective of {protagonist_name}.
For describing your own body movements, wrap your description in '*'.
Do not change roles!
Do not speak from the perspective of {storyteller_name}.
Do not forget to finish speaking by saying, 'It is your turn, {storyteller_name}.'
Do not add anything else.
Remember you are the protagonist, {protagonist_name}.
Stop speaking the moment you finish speaking from your perspective.
"""
    )
)

storyteller_system_message = SystemMessage(
    content=(
        f"""{game_description}
Never forget you are the storyteller, {storyteller_name}, and I am the protagonist, {protagonist_name}. 
Your character description is as follows: {storyteller_description}.
I will propose actions I plan to take and you will explain what happens when I take those actions.
Speak in the first person from the perspective of {storyteller_name}.
For describing your own body movements, wrap your description in '*'.
Do not change roles!
Do not speak from the perspective of {protagonist_name}.
Do not forget to finish speaking by saying, 'It is your turn, {protagonist_name}.'
Do not add anything else.
Remember you are the storyteller, {storyteller_name}.
Stop speaking the moment you finish speaking from your perspective.
"""
    )
)

## 使用 LLM 创建详细的任务描述

This guide will walk you through using a large language model (LLM) to create an elaborate quest description.

本指南将引导您使用大型语言模型 (LLM) 创建详细的任务描述。

### Prerequisites

### 先决条件

*   **An LLM access:** You can use OpenAI's API, or any other LLM provider.
*   **LLM 访问权限：** 您可以使用 OpenAI 的 API，或任何其他 LLM 提供商。
*   **Basic understanding of prompt engineering:** Knowing how to craft effective prompts will help you get the best results.
*   **提示工程基础知识：** 了解如何构建有效的提示将帮助您获得最佳结果。

### Steps

### 步骤

1.  **Define the core quest idea:** What is the basic premise of your quest? Who is the player character? What is the main objective?
    1.  **定义核心任务创意：** 您的任务的基本前提是什么？玩家角色是谁？主要目标是什么？

    *Example:* A knight must retrieve a stolen artifact from a dragon's lair.
    *示例：* 一位骑士必须从龙穴中取回被盗的圣物。

2.  **Craft a detailed prompt for the LLM:** The more detail you provide, the better the LLM can understand your needs. Include information about:
    2.  **为 LLM 创建详细的提示：** 您提供的细节越多，LLM 就越能理解您的需求。包含以下信息：

    *   **Quest Giver:** Who is giving the quest? What is their personality and motivation?
        *   **任务发布者：** 谁在发布任务？他们的个性和动机是什么？
    *   **Setting:** Where does the quest take place? Describe the environment, atmosphere, and any notable landmarks.
        *   **场景：** 任务在哪里进行？描述环境、氛围和任何显著的地标。
    *   **Objective:** What exactly does the player need to do? Break it down into smaller steps if necessary.
        *   **目标：** 玩家具体需要做什么？如有必要，将其分解为更小的步骤。
    *   **Challenges/Obstacles:** What difficulties will the player face? (e.g., enemies, puzzles, environmental hazards).
        *   **挑战/障碍：** 玩家将面临哪些困难？（例如，敌人、谜题、环境危害）。
    *   **Rewards:** What will the player receive upon completion? (e.g., gold, experience, items, reputation).
        *   **奖励：** 完成后玩家将获得什么？（例如，金币、经验值、物品、声望）。
    *   **Lore/Backstory:** Any relevant history or context that enriches the quest.
        *   **传说/背景故事：** 任何相关的历史或背景信息，以丰富任务内容。
    *   **Tone/Style:** What kind of feeling should the description evoke? (e.g., epic, mysterious, humorous).
        *   **语气/风格：** 描述应该唤起什么样的感觉？（例如，史诗般的、神秘的、幽默的）。

    *Example Prompt:*
    *示例提示：*

    ```
    Generate an elaborate quest description for a fantasy RPG.

    Quest Giver: Elara Meadowlight, a wise but weary elven herbalist living in a secluded cottage near the Whispering Woods. She is concerned about the unnatural blight spreading from the old ruins.

    Setting: The Whispering Woods, an ancient forest known for its eerie silence and glowing flora. The quest will lead to the Sunken Temple ruins, a moss-covered, crumbling structure half-submerged in a murky swamp.

    Objective: Retrieve the 'Heartstone of Eldoria', a magical gem that Elara believes can cleanse the blight. The Heartstone is said to be guarded by ancient spirits within the Sunken Temple.

    Challenges:
    - Navigate the treacherous Whispering Woods, avoiding territorial sprites and illusionary paths.
    - Solve a riddle posed by a spectral guardian at the entrance of the Sunken Temple.
    - Defeat corrupted swamp creatures that have been twisted by the blight.
    - Overcome a magical barrier protecting the Heartstone.

    Rewards:
    - The 'Amulet of Verdant Growth' (provides minor healing over time).
    - 500 experience points.
    - Increased reputation with the Elven community.
    - Elara's gratitude and a potent healing potion.

    Lore/Backstory: The Sunken Temple was once a place of healing, but was corrupted centuries ago by a dark ritual. The blight is a manifestation of this lingering corruption. The Heartstone of Eldoria was the temple's original power source.

    Tone/Style: Mysterious, slightly melancholic, with a sense of ancient magic and creeping danger.
    ```

    ```
    为奇幻角色扮演游戏生成一个详细的任务描述。

    任务发布者：艾拉·草光 (Elara Meadowlight)，一位居住在低语森林附近隐蔽小屋里的睿智但疲惫的精灵草药师。她对从古老遗迹中蔓延开来的不自然枯萎病感到担忧。

    场景：低语森林，一片以其诡异的寂静和发光的植物而闻名的古老森林。任务将引导玩家前往沉没神庙遗迹，这是一个长满苔藓、摇摇欲坠的建筑，半淹没在浑浊的沼泽中。

    目标：取回“艾尔多利亚之心石”（Heartstone of Eldoria），艾拉相信这颗魔法宝石可以净化枯萎病。据说之心石受到沉没神庙内古老灵魂的守护。

    挑战：
    - 穿越危险的低语森林，避开领地意识强的精怪和幻象路径。
    - 解开沉没神庙入口处一位幽灵守护者提出的谜语。
    - 击败被枯萎病扭曲的腐化沼泽生物。
    - 克服保护之心石的魔法屏障。

    奖励：
    - “翠绿生长护符”（Amulet of Verdant Growth）（提供少量持续治疗效果）。
    - 500 点经验值。
    - 与精灵社区的声望提升。
    - 艾拉的感激和一个强效治疗药水。

    传说/背景故事：沉没神庙曾是一个疗愈之地，但在几个世纪前被一个黑暗仪式所腐化。枯萎病是这种挥之不去的腐化的体现。艾尔多利亚之心石是神庙最初的能量源。

    语气/风格：神秘、略带忧郁，带有古老魔法和潜伏危险的感觉。
    ```

3.  **Submit the prompt to the LLM and review the output:** Read through the generated description. Does it capture the essence of your idea? Is it engaging and informative?
    3.  **将提示提交给 LLM 并审查输出：** 阅读生成的描述。它是否抓住了您想法的精髓？它是否引人入胜且信息丰富？

4.  **Iterate and refine:** If the output isn't quite right, adjust your prompt and try again. You might need to be more specific about certain elements or ask the LLM to emphasize particular aspects.
    4.  **迭代和完善：** 如果输出不完全符合要求，请调整您的提示并重试。您可能需要更具体地说明某些元素，或要求 LLM 强调特定方面。

    *Example Refinement:* If the description is too generic, you could add: "Emphasize the feeling of decay and the struggle against nature's corruption in the Whispering Woods."
    *示例完善：* 如果描述过于笼统，您可以添加：“强调低语森林中腐朽的感觉以及与自然腐化的斗争。”

### Example LLM Output (based on the prompt above)

### LLM 输出示例（基于上述提示）

```markdown
## The Blight of the Sunken Temple

**Quest Giver:** Elara Meadowlight, the reclusive elven herbalist of Whisperwood.

**Location:** The Whispering Woods and the Sunken Temple Ruins.

**Recommended Level:** 5-7

---

**Description:**

A creeping, unnatural blight has begun to poison the ancient heart of the Whispering Woods. The vibrant, glowing flora dims, and an unsettling silence has fallen over the once-lively forest. Elara Meadowlight, a wise elven herbalist who has lived in seclusion for decades, has observed this decay with growing alarm. She believes the source of this corruption lies deep within the forgotten **Sunken Temple**, a crumbling edifice half-swallowed by the murky swamp to the west.

Elara speaks of a legend – the **Heartstone of Eldoria**, a gem of immense purity that once powered the temple as a sanctuary of healing. She fears it has been corrupted or its power has been twisted, allowing the blight to spread. She implores you to venture into the Whispering Woods, brave the treacherous paths, and seek out the Sunken Temple.

Your task is to infiltrate the ruins, overcome the ancient guardians and corrupted denizens twisted by the blight, and retrieve the Heartstone of Eldoria. Only by returning it to Elara can the cleansing ritual be performed, and the Whispering Woods be saved from this encroaching darkness.

**Objectives:**

1.  **Seek out Elara Meadowlight** in her cottage on the edge of the Whispering Woods.
2.  **Journey through the Whispering Woods**, navigating its illusions and avoiding its territorial inhabitants.
3.  **Locate the Sunken Temple Ruins.**
4.  **Solve the riddle of the spectral guardian** at the temple's entrance.
5.  **Brave the depths of the temple**, battling corrupted creatures and overcoming the blight's influence.
6.  **Retrieve the Heartstone of Eldoria** from its protected chamber.
7.  **Return the Heartstone to Elara Meadowlight.**

**Challenges:**

*   **Whispering Woods Navigation:** The woods are known for their disorienting illusions and aggressive sprites. Follow Elara's crude map carefully.
*   **Spectral Guardian's Riddle:** A spectral guardian protects the temple entrance. Its riddle must be answered correctly to gain passage.
*   **Swamp Creatures:** The blight has mutated the local fauna into aggressive, twisted forms.
*   **The Temple's Corruption:** The very air within the temple feels heavy and diseased. A magical barrier protects the Heartstone.

**Rewards:**

*   **Amulet of Verdant Growth:** A finely crafted amulet that slowly regenerates the wearer's vitality.
*   **500 Experience Points**
*   **Elven Gratitude:** Your reputation with the elves of the region will increase significantly.
*   **Potent Healing Potion:** A powerful concoction crafted by Elara herself.

*May the ancient spirits guide your path.*
```

```markdown
## 沉没神庙的枯萎病

**任务发布者：** 艾拉·草光 (Elara Meadowlight)，低语森林隐居的精灵草药师。

**地点：** 低语森林和沉没神庙遗迹。

**推荐等级：** 5-7

---

**描述：**

一种蔓延的、不自然的枯萎病开始毒害低语森林的古老核心。充满活力的发光植物变得黯淡，曾经生机勃勃的森林笼罩着一种令人不安的寂静。艾拉·草光，一位隐居了数十年的睿智精灵草药师，正日益警惕地观察着这种衰败。她认为这种腐化的源头在于被遗忘的 **沉没神庙** 深处，这是一个位于西部浑浊沼泽中的半淹没的摇摇欲坠的建筑。

艾拉讲述了一个传说——**艾尔多利亚之心石**，一颗纯净无瑕的宝石，曾作为疗愈圣地驱动着神庙。她担心它已被腐化或其力量已被扭曲，导致枯萎病蔓延。她恳求您冒险进入低语森林，勇敢地穿越危险的路径，并寻找沉没神庙。

您的任务是潜入遗迹，克服古老的守护者和被枯萎病扭曲的腐化居民，并取回艾尔多利亚之心石。只有将其带回给艾拉，才能进行净化仪式，并将低语森林从这迫近的黑暗中拯救出来。

**目标：**

1.  在低语森林边缘的草屋中**寻找艾拉·草光**。
2.  **穿越低语森林**，在其中迷宫般的幻象中穿行，并避开其领地意识强的居民。
3.  **找到沉没神庙遗迹。**
4.  **解开神庙入口处幽灵守护者的谜语。**
5.  **勇敢深入神庙**，与腐化的生物战斗，并克服枯萎病的影响。
6.  **从其受保护的密室中取回艾尔多利亚之心石。**
7.  **将之心石带回给艾拉·草光。**

**挑战：**

*   **低语森林导航：** 这片森林以其令人迷失方向的幻象和具有攻击性的精怪而闻名。请仔细遵循艾拉粗略绘制的地图。
*   **幽灵守护者的谜语：** 一位幽灵守护者守护着神庙入口。必须正确回答它的谜语才能通过。
*   **沼泽生物：** 枯萎病已将当地的动物群变异成具有攻击性、扭曲的形态。
*   **神庙的腐化：** 神庙内的空气本身就感觉沉重而充满疾病。一道魔法屏障保护着之心石。

**奖励：**

*   **翠绿生长护符：** 一枚制作精美的护符，能缓慢恢复佩戴者的活力。
*   **500 点经验值**
*   **精灵的感激：** 您在该地区精灵中的声望将显著提高。
*   **强效治疗药水：** 由艾拉亲手制作的强效药剂。

*愿古老灵魂指引你的道路。*

In [8]:
quest_specifier_prompt = [
    SystemMessage(content="You can make a task more specific."),
    HumanMessage(
        content=f"""{game_description}
        
        You are the storyteller, {storyteller_name}.
        Please make the quest more specific. Be creative and imaginative.
        Please reply with the specified quest in {word_limit} words or less. 
        Speak directly to the protagonist {protagonist_name}.
        Do not add anything else."""
    ),
]
specified_quest = ChatOpenAI(temperature=1.0)(quest_specifier_prompt).content

print(f"Original quest:\n{quest}\n")
print(f"Detailed quest:\n{specified_quest}\n")

Original quest:
Find all of Lord Voldemort's seven horcruxes.

Detailed quest:
Harry, you must venture to the depths of the Forbidden Forest where you will find a hidden labyrinth. Within it, lies one of Voldemort's horcruxes, the locket. But beware, the labyrinth is heavily guarded by dark creatures and spells, and time is running out. Can you find the locket before it's too late?



## 主循环

The main loop is the core of the application. It's responsible for **polling** for new events and **dispatching** them to the appropriate handlers.

主循环是应用程序的核心。它负责**轮询**新事件并将其**分派**给相应的处理程序。

```jsx
function mainLoop() {
  // Poll for new events
  const events = pollEvents();

  // Dispatch events to handlers
  for (const event of events) {
    handleEvent(event);
  }

  // Request the next animation frame
  requestAnimationFrame(mainLoop);
}

// Start the main loop
mainLoop();
```

### Event Polling

The `pollEvents()` function is responsible for retrieving new events from the system. This could include user input, network messages, or other system events.

### 事件轮询

`pollEvents()` 函数负责从系统中检索新事件。这可能包括用户输入、网络消息或其他系统事件。

### Event Dispatching

The `handleEvent(event)` function is responsible for processing a single event. It will determine the type of event and call the appropriate handler function.

### 事件分派

`handleEvent(event)` 函数负责处理单个事件。它将确定事件的类型并调用相应的处理程序函数。

### Animation Frames

The `requestAnimationFrame(callback)` function is a browser API that tells the browser you wish to perform an animation and requests that the browser call a specified function to update an animation before the next repaint. This is the preferred way to perform animations in the browser, as it allows the browser to optimize the animation and ensure it runs smoothly.

### 动画帧

`requestAnimationFrame(callback)` 函数是一个浏览器 API，它告诉浏览器您希望执行动画，并请求浏览器在下一次重绘之前调用指定函数来更新动画。这是在浏览器中执行动画的首选方式，因为它允许浏览器优化动画并确保其流畅运行。

In [16]:
protagonist = DialogueAgent(
    name=protagonist_name,
    system_message=protagonist_system_message,
    model=ChatOpenAI(temperature=0.2),
)
storyteller = DialogueAgent(
    name=storyteller_name,
    system_message=storyteller_system_message,
    model=ChatOpenAI(temperature=0.2),
)

In [17]:
def select_next_speaker(step: int, agents: List[DialogueAgent]) -> int:
    idx = step % len(agents)
    return idx

In [18]:
max_iters = 6
n = 0

simulator = DialogueSimulator(
    agents=[storyteller, protagonist], selection_function=select_next_speaker
)
simulator.reset()
simulator.inject(storyteller_name, specified_quest)
print(f"({storyteller_name}): {specified_quest}")
print("\n")

while n < max_iters:
    name, message = simulator.step()
    print(f"({name}): {message}")
    print("\n")
    n += 1

(Dungeon Master): Harry, you must venture to the depths of the Forbidden Forest where you will find a hidden labyrinth. Within it, lies one of Voldemort's horcruxes, the locket. But beware, the labyrinth is heavily guarded by dark creatures and spells, and time is running out. Can you find the locket before it's too late?


(Harry Potter): I take a deep breath and ready my wand. I know this won't be easy, but I'm determined to find that locket and destroy it. I start making my way towards the Forbidden Forest, keeping an eye out for any signs of danger. As I enter the forest, I cast a protective spell around myself and begin to navigate through the trees. I keep my wand at the ready, prepared for any surprises that may come my way. It's going to be a long and difficult journey, but I won't give up until I find that horcrux. It is your turn, Dungeon Master.


(Dungeon Master): As you make your way through the Forbidden Forest, you hear the rustling of leaves and the snapping of twigs. S